# 12 — Ajout de PDF et mise à jour incrémentale

Décrit le chemin d'un PDF ajouté dans l'application et vérifie les documents déjà enregistrés sans reconstruire tout l'index.

> Vérifiez toujours les drapeaux `RUN_*` avant l'exécution. Pour cette reconstruction propre, les traitements locaux nécessaires des étapes 02 à 07 sont déjà activés ; les collectes réseau et évaluations GPU facultatives restent désactivées.


In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.yaml').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Impossible de trouver config.yaml et src/.')

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Projet :', PROJECT_ROOT)
print('Python :', sys.executable)
print('Données actives :', PROJECT_ROOT / 'data')


## Chemin d'un document ajouté

`Upload PDF → validation et empreinte → extraction du texte → nettoyage → chunks parent-enfant → embeddings des nouveaux chunks → mise à jour LanceDB → reconstruction BM25 → document interrogeable`

L'index dense ne recalcule pas les milliers de chunks inchangés. BM25, beaucoup plus léger, est reconstruit pour rester cohérent.


In [ ]:
import sqlite3
import pandas as pd
from config import load_config

config = load_config(PROJECT_ROOT / 'config.yaml')
database_path = config.project.data_path / 'app' / 'rag_ensea.sqlite3'
uploads_path = config.project.data_path / 'uploads' / 'pdfs'
print('Base application :', database_path)
print('Dossier PDF :', uploads_path)
if database_path.exists():
    with sqlite3.connect(database_path) as connection:
        documents = pd.read_sql_query('SELECT id, filename, title, status, page_count, chunk_count, created_at FROM documents ORDER BY created_at DESC', connection)
    display(documents)


In [ ]:
import requests

RUN_UPLOAD_TEST = False
PDF_TO_UPLOAD = PROJECT_ROOT / 'chemin' / 'vers' / 'document.pdf'
if RUN_UPLOAD_TEST:
    if not PDF_TO_UPLOAD.exists():
        raise FileNotFoundError(PDF_TO_UPLOAD)
    with PDF_TO_UPLOAD.open('rb') as handle:
        response = requests.post(
            'http://127.0.0.1:8000/api/documents',
            files={'file': (PDF_TO_UPLOAD.name, handle, 'application/pdf')},
            data={'title': PDF_TO_UPLOAD.stem, 'category': 'rapport'},
            timeout=120,
        )
    response.raise_for_status()
    display(response.json())
else:
    print('Upload de test désactivé. Utilisez normalement la bibliothèque de l’application.')


## Contrôles

Vérifiez le nombre de pages, le nombre de chunks, le statut `ready`, puis posez une question dont la réponse figure uniquement dans le nouveau PDF. Une empreinte empêche l'ajout accidentel du même fichier deux fois.
